# 예제 01. 왜 도구가 필요한가 — 모델은 계산을 못 한다

전산실험 2 · AI Agent 만들기 · **5주차** · 예상 소요 30분

## 목표
- 모델이 계산을 얼마나 틀리는지 **측정**한다
- 자릿수·연산 종류에 따라 오차가 어떻게 달라지는지 본다
- **자신 있게 틀린다**는 것이 왜 최악인지 확인한다
- 같은 계산을 도구로 하면 어떻게 되는지 비교한다

> 1주차 연습문제에서 "LLM에게 난수를 뽑게 하지 마라"고 했습니다.
> 오늘 그 원칙을 일반화합니다.

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)"""
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 --------------------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용."""
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다."""
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과."""
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 ------------------------------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다."""
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다."""
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다."""
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile structured.py
"""전산실험2 3주차 : 구조화 출력 — 파싱 · 검증 · 재시도.

소형 모델은 "JSON으로만 답하라"고 지시해도 자주 어긴다.
코드펜스로 감싸거나, 인사를 덧붙이거나, 따옴표를 빠뜨린다.

이 모듈은 세 겹의 방어를 제공한다.

    1) 프롬프트로 형식을 강제하고        (build_json_prompt)
    2) 지저분한 응답에서 JSON을 긁어내고  (extract_json)
    3) 스키마로 검증한 뒤, 실패하면       (validate)
       오류를 되먹여 다시 요구한다        (ask_json)

사용 예:
    from kmu_llm import LLM
    from structured import ask_json

    schema = {
        "required": ["test", "reason"],
        "types": {"test": str, "reason": str},
        "allowed": {"test": ["t-test", "anova", "chi-square"]},
    }
    result = ask_json("두 그룹의 평균 비교에 쓸 검정은?", schema=schema)
    print(result["test"])
"""

from __future__ import annotations

import json
import re
from typing import Any

__all__ = [
    "JSONParseError",
    "SchemaError",
    "extract_json",
    "parse_json",
    "validate",
    "build_json_prompt",
    "ask_json",
]


class JSONParseError(ValueError):
    """모델 응답에서 JSON을 뽑아내지 못했을 때."""


class SchemaError(ValueError):
    """JSON은 얻었지만 요구한 구조가 아닐 때."""


# --------------------------------------------------------------------------
# 1) 지저분한 응답에서 JSON 긁어내기
# --------------------------------------------------------------------------
_FENCE = re.compile(r"```(?:json|python)?\s*(.*?)```", re.DOTALL)


def extract_json(text: str) -> str:
    """모델 응답에서 JSON 문자열만 뽑아낸다.

    소형 모델이 실제로 내놓는 형태들:
        ```json\\n{...}\\n```
        네, 알겠습니다! {...}
        {...}\\n도움이 되었길 바랍니다.

    Returns
    -------
    str : JSON 으로 보이는 부분

    Raises
    ------
    JSONParseError : 중괄호/대괄호 쌍을 찾지 못한 경우
    """
    if not text or not text.strip():
        raise JSONParseError("응답이 비어 있습니다.")

    # (1) 코드펜스가 있으면 그 안쪽을 우선 사용
    fence = _FENCE.search(text)
    if fence:
        text = fence.group(1)

    # (2) 가장 바깥쪽 { } 또는 [ ] 를 찾는다
    for opener, closer in (("{", "}"), ("[", "]")):
        start = text.find(opener)
        end = text.rfind(closer)
        if start != -1 and end > start:
            return text[start : end + 1].strip()

    raise JSONParseError(
        f"JSON 구조를 찾지 못했습니다. 응답 앞부분: {text[:100]!r}"
    )


def parse_json(text: str) -> Any:
    """extract_json + json.loads. 실패하면 JSONParseError."""
    snippet = extract_json(text)
    try:
        return json.loads(snippet)
    except json.JSONDecodeError as exc:
        # 흔한 사고: 마지막 항목 뒤의 쉼표(trailing comma)
        repaired = re.sub(r",\s*([}\]])", r"\1", snippet)
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass
        raise JSONParseError(
            f"JSON 파싱 실패 ({exc.msg}, {exc.lineno}행 {exc.colno}열). "
            f"문제 부분: {snippet[max(0, exc.pos - 30) : exc.pos + 30]!r}"
        ) from exc


# --------------------------------------------------------------------------
# 2) 스키마 검증
# --------------------------------------------------------------------------
def validate(data: Any, schema: dict | None) -> list[str]:
    """아주 단순한 스키마 검증기. 문제점 목록을 돌려준다 (빈 목록 = 통과).

    스키마 형식::

        {
            "required": ["test", "reason"],          # 반드시 있어야 할 키
            "types":    {"alpha": float, "n": int},  # 값의 파이썬 타입
            "allowed":  {"test": ["t-test", ...]},   # 허용되는 값 목록
            "range":    {"alpha": (0.0, 1.0)},       # 수치 범위 (닫힌 구간)
        }

    pydantic 같은 라이브러리를 쓸 수도 있지만, 여기서는 검증이 무엇을 하는
    일인지 보이도록 직접 구현한다.
    """
    if schema is None:
        return []

    problems: list[str] = []

    if not isinstance(data, dict):
        return [f"최상위가 객체(dict)가 아니라 {type(data).__name__} 입니다."]

    for key in schema.get("required", []):
        if key not in data:
            problems.append(f"필수 키 '{key}' 가 없습니다.")

    for key, expected in schema.get("types", {}).items():
        if key not in data:
            continue
        value = data[key]
        # int 는 float 자리에 들어와도 허용한다
        if expected is float and isinstance(value, int) and not isinstance(value, bool):
            continue
        if not isinstance(value, expected):
            problems.append(
                f"'{key}' 는 {expected.__name__} 여야 하는데 "
                f"{type(value).__name__} 입니다 (값: {value!r})."
            )

    for key, options in schema.get("allowed", {}).items():
        if key in data and data[key] not in options:
            problems.append(
                f"'{key}' 의 값 {data[key]!r} 는 허용 목록에 없습니다. "
                f"가능한 값: {options}"
            )

    for key, (low, high) in schema.get("range", {}).items():
        if key not in data:
            continue
        value = data[key]
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            problems.append(f"'{key}' 가 숫자가 아니어서 범위를 확인할 수 없습니다.")
        elif not (low <= value <= high):
            problems.append(f"'{key}' 의 값 {value} 가 [{low}, {high}] 범위 밖입니다.")

    return problems


# --------------------------------------------------------------------------
# 3) 프롬프트로 형식 강제하기
# --------------------------------------------------------------------------
def build_json_prompt(prompt: str, schema: dict | None = None, example: Any = None) -> str:
    """JSON 출력을 유도하는 지시문을 프롬프트에 덧붙인다.

    소형 모델에서 실제로 효과가 있었던 순서대로 넣는다.
    1) 출력 예시  2) 금지 사항 명시  3) 시작·끝 문자 지정
    """
    parts = [prompt, "", "## 출력 형식"]

    if example is not None:
        parts.append("아래와 정확히 같은 형식의 JSON으로만 답하세요.")
        parts.append(json.dumps(example, ensure_ascii=False, indent=2))
    elif schema and schema.get("required"):
        keys = ", ".join(f'"{k}"' for k in schema["required"])
        parts.append(f"다음 키를 가진 JSON 객체로만 답하세요: {keys}")

    if schema:
        for key, options in schema.get("allowed", {}).items():
            parts.append(f'- "{key}" 는 다음 중 하나여야 합니다: {options}')

    parts += [
        "",
        "규칙:",
        "- 설명, 인사, 사족을 쓰지 마세요.",
        "- 코드블록 표시(백틱)를 쓰지 마세요.",
        "- 응답은 반드시 { 로 시작해서 } 로 끝나야 합니다.",
    ]
    return "\n".join(parts)


# --------------------------------------------------------------------------
# 4) 파싱 · 검증 · 재시도를 합친 것
# --------------------------------------------------------------------------
def ask_json(
    prompt: str,
    system: str | None = None,
    *,
    schema: dict | None = None,
    example: Any = None,
    n_retry: int = 3,
    llm: Any = None,
    verbose: bool = False,
    **kw: Any,
) -> Any:
    """모델에게 JSON을 받아 파싱·검증한 결과를 돌려준다.

    실패하면 **오류 메시지를 모델에게 되돌려주고 다시 요구한다.**
    이 되먹임 구조가 6주차 ReAct 루프의 핵심 아이디어와 같다.

    Parameters
    ----------
    schema  : validate() 형식의 스키마
    example : 출력 예시 (few-shot). 소형 모델에서 가장 효과가 크다
    n_retry : 최대 시도 횟수
    llm     : LLM 인스턴스. 없으면 kmu_llm.LLM() 을 만든다
    verbose : 시도 과정을 출력

    Raises
    ------
    JSONParseError / SchemaError : n_retry 회 모두 실패한 경우
    """
    if llm is None:
        from kmu_llm import LLM

        llm = LLM()

    kw.setdefault("temperature", 0.0)  # 형식을 지켜야 하므로 낮게

    messages: list[dict[str, str]] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": build_json_prompt(prompt, schema, example)})

    last_error: Exception | None = None

    for attempt in range(1, n_retry + 1):
        raw = llm.chat(messages, **kw)
        if verbose:
            print(f"[시도 {attempt}] 원문: {raw[:120]!r}")

        try:
            data = parse_json(raw)
        except JSONParseError as exc:
            last_error = exc
            feedback = (
                f"방금 응답을 JSON으로 읽을 수 없었습니다. 오류: {exc}\n"
                "설명 없이 JSON 객체만 다시 출력하세요."
            )
        else:
            problems = validate(data, schema)
            if not problems:
                if verbose:
                    print(f"[시도 {attempt}] 성공")
                return data
            last_error = SchemaError("; ".join(problems))
            feedback = (
                "JSON 형식은 맞지만 내용에 문제가 있습니다:\n"
                + "\n".join(f"- {p}" for p in problems)
                + "\n이 문제들을 고쳐서 JSON 객체만 다시 출력하세요."
            )

        if verbose:
            print(f"[시도 {attempt}] 실패 -> 되먹임: {last_error}")

        # 실패한 응답과 오류를 대화에 남겨서 모델이 스스로 고치게 한다
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": feedback})

    raise type(last_error)(  # type: ignore[misc]
        f"{n_retry}회 시도 모두 실패했습니다. 마지막 오류: {last_error}"
    )


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

---
## 1. 모델에게 평균을 계산시켜 본다

숫자 30개를 주고 평균·표준편차·중앙값을 구하게 합니다.
정답은 numpy 가 알고 있습니다.

In [ ]:
import numpy as np
import pandas as pd
from structured import ask_json

rng = np.random.default_rng(42)

계산_스키마 = {
    "required": ["mean", "sd", "median"],
    "types": {"mean": float, "sd": float, "median": float},
}

def 모델계산(values):
    나열 = ", ".join(str(v) for v in values)
    return ask_json(
        f"다음 30개 숫자의 평균, 표준편차(표본, ddof=1), 중앙값을 계산하세요.\n{나열}",
        schema=계산_스키마,
        example={"mean": 0.0, "sd": 0.0, "median": 0.0},
        llm=llm, n_retry=2, max_tokens=300,
    )

values = rng.integers(10, 100, 30).tolist()
print("숫자:", values)
print()
print("모델:", 모델계산(values))
print("정답:", {"mean": round(float(np.mean(values)), 4),
               "sd": round(float(np.std(values, ddof=1)), 4),
               "median": round(float(np.median(values)), 4)})

한 번으로는 알 수 없습니다. **20회 반복해서 오차 분포를 봅시다.**

In [ ]:
기록 = []
for i in range(20):
    v = rng.integers(10, 100, 30).tolist()
    정답 = {"mean": float(np.mean(v)), "sd": float(np.std(v, ddof=1)),
            "median": float(np.median(v))}
    try:
        답 = 모델계산(v)
    except Exception:
        continue
    for k in ("mean", "sd", "median"):
        기록.append({
            "통계량": k,
            "정답": 정답[k],
            "모델": float(답[k]),
            "상대오차": abs(float(답[k]) - 정답[k]) / max(abs(정답[k]), 1e-9),
        })
    print(".", end="", flush=True)
print(" 완료")

계산df = pd.DataFrame(기록)
계산df.groupby("통계량")["상대오차"].describe()[["count", "mean", "50%", "max"]].round(4)

In [ ]:
# 오차 1% 이내를 '맞았다'고 보면?
정확 = 계산df.assign(맞음=계산df["상대오차"] < 0.01).groupby("통계량")["맞음"].mean()
print("상대오차 1% 이내 비율")
print((정확 * 100).round(1).to_string())
print()
print("-> 평균은 그럭저럭, 표준편차는 무너집니다.")
print("   제곱하고 더하고 나누고 제곱근을 씌우는 다단계 계산이기 때문입니다.")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for k in ["mean", "sd", "median"]:
    부분 = 계산df[계산df["통계량"] == k]["상대오차"]
    ax.scatter([k] * len(부분), 부분, alpha=0.6, s=40)
ax.axhline(0.01, ls="--", c="red", label="1% error")
ax.set_yscale("symlog", linthresh=1e-4)
ax.set_ylabel("relative error (log scale)")
ax.set_title("LLM arithmetic accuracy (30 numbers, 20 trials)")
ax.legend()
plt.tight_layout(); plt.show()

---
## 2. 자릿수를 늘리면?

작은 정수는 학습 데이터에 워낙 많이 나와서 그럭저럭 맞힙니다.
소수점과 자릿수를 늘려 봅시다.

In [ ]:
조건들 = {
    "2자리 정수":     lambda: rng.integers(10, 100, 20).tolist(),
    "4자리 정수":     lambda: rng.integers(1000, 9999, 20).tolist(),
    "소수점 2자리":   lambda: np.round(rng.uniform(10, 100, 20), 2).tolist(),
    "소수점 4자리":   lambda: np.round(rng.uniform(10, 100, 20), 4).tolist(),
}

평균_스키마 = {"required": ["mean"], "types": {"mean": float}}
결과 = []
for 이름, gen in 조건들.items():
    print(f"{이름:14s}", end=" ")
    for _ in range(8):
        v = gen()
        try:
            답 = ask_json(f"다음 숫자들의 평균을 구하세요.\n{v}",
                         schema=평균_스키마, example={"mean": 0.0},
                         llm=llm, n_retry=2, max_tokens=150)
            오차 = abs(float(답["mean"]) - float(np.mean(v))) / abs(float(np.mean(v)))
        except Exception:
            오차 = np.nan
        결과.append({"조건": 이름, "상대오차": 오차})
        print(".", end="", flush=True)
    print()

자릿수df = pd.DataFrame(결과)
자릿수df.groupby("조건", sort=False)["상대오차"].agg(
    평균오차="mean", 중앙값="median", 정확비율=lambda s: (s < 0.01).mean()
).round(4)

---
## 3. 진짜 문제는 '자신 있게 틀린다'는 것

모델은 "제가 계산에 약해서 확실하지 않습니다"라고 말하지 않습니다.
**그럴듯한 숫자를 확신에 차서 내놓습니다.**

In [ ]:
v = np.round(rng.uniform(100, 200, 25), 3).tolist()
답 = llm.ask(
    f"다음 숫자들의 표준편차(ddof=1)를 계산하고, 계산 과정과 확신도를 설명하세요.\n{v}",
    temperature=0.0, max_tokens=500,
)
print(답)
print()
print("=" * 60)
print("실제 정답:", round(float(np.std(v, ddof=1)), 4))

> 위 응답에서 모델이 **확신에 찬 어조**로 말하는지 보십시오.
> 통계 분석 도구에서 이것은 최악의 실패 방식입니다.
>
> 3주차에서 말한 원칙이 여기서도 적용됩니다.
> **조용히 틀리느니 명확하게 실패하는 편이 낫습니다.**
> 그런데 모델은 조용히 틀립니다. 그래서 애초에 시키지 않습니다.

---
## 4. 도구로 하면

같은 계산을 numpy 에게 시킵니다. 오차는 당연히 0 입니다.

In [ ]:
def 도구계산(values):
    a = np.asarray(values, dtype=float)
    return {"mean": round(float(a.mean()), 4),
            "sd": round(float(a.std(ddof=1)), 4),
            "median": round(float(np.median(a)), 4)}

v = rng.integers(10, 100, 30).tolist()
print("도구:", 도구계산(v))
print("정답:", {"mean": round(float(np.mean(v)), 4),
               "sd": round(float(np.std(v, ddof=1)), 4),
               "median": round(float(np.median(v)), 4)})
print()
print("오차 0. 당연합니다. numpy 가 계산했으니까요.")

### 역할 분담

| 모델이 잘하는 것 | 도구가 하는 것 |
|---|---|
| 어떤 검정을 쓸지 **판단** | 검정을 실제로 **수행** |
| 열 이름의 의미 파악 | 기술통계 계산 |
| 결과를 문장으로 **해석** | p값 계산 |
| 다음에 뭘 할지 **결정** | 그림 그리기 |

**모델은 판단하고, 도구는 계산합니다.**

이렇게 나누면 검증도 가능해집니다.
모델이 고른 도구와 인자가 맞는지는 우리가 확인할 수 있습니다.
반면 모델이 내놓은 p값이 맞는지 확인하려면 어차피 다시 계산해야 합니다.

---
## 연습문제

**문제 1.** 자릿수(2, 3, 4, 5)와 상대오차의 관계를 회귀로 확인하시오.
자릿수가 하나 늘 때 오차는 몇 배가 되는가?

**문제 2.** 모델에게 **계산 과정을 단계별로 쓰게 한 뒤**(chain-of-thought)
최종 답을 받으면 정확도가 오르는가? 20회씩 비교하시오.

**문제 3.** 숫자 개수를 10, 30, 100 으로 바꿔 가며 평균 계산 정확도를 재시오.
개수가 늘면 왜 더 틀리는가?

**문제 4.** 모델에게 "계산이 어려우면 모른다고 답하라"고 지시하면
실제로 모른다고 하는가? 20회 시도해 보시오.

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 2.** 대개 **오릅니다.** 단계를 나누면 각 단계가 학습 데이터에서 본
패턴에 가까워지기 때문입니다. 하지만 두 가지 대가가 있습니다.
토큰이 크게 늘고(4주차 예산 문제), 여전히 100% 가 아닙니다.

**정확도가 올라도 도구를 쓰는 편이 낫습니다.** 도구는 오차가 0이고 토큰도
안 씁니다. chain-of-thought 는 **계산이 아니라 판단**에 쓰는 기법입니다.

**문제 4.** 대부분 모른다고 하지 않고 답을 내놓습니다. 이것이 핵심입니다.
모델은 자기가 계산을 못 한다는 것을 모릅니다.
**"모르면 모른다고 하라"는 지시로는 이 문제를 해결할 수 없습니다.**
계산 자체를 맡기지 않는 것이 유일한 해법입니다.

</details>

---
## 정리

- 모델은 계산기가 아니라 **다음 토큰 예측기**다
- 다단계 계산(표준편차)일수록, 자릿수가 클수록 더 틀린다
- **자신 있게 틀린다.** 이것이 최악이다
- **모델은 판단, 도구는 계산**

**다음 노트북** → [`02_tool_registry.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week05/02_tool_registry.ipynb)
도구를 등록하고 부르는 틀을 만듭니다.